## Train the model

### 0 - Sagemaker Configuration

In [ ]:
import sagemaker
from sagemaker.image_uris import retrieve
from sagemaker.serializers import CSVSerializer

In [ ]:
# Region
region = sagemaker.Session().boto_region_name
print("AWS Region: {}".format(region))

# Role
role = sagemaker.get_execution_role()
print("RoleArn: {}".format(role))

# Sagemaker session
session = sagemaker.Session()

### 1 - Model selection 

In [ ]:
# We need the XGBoost model container. The container name changes depending on the region and model.
# To help with this, we use the get_image_uri function.
container = retrieve('xgboost', region, version='latest')

In [ ]:
s3_folder_bucket = "sentiment-classification-bucket"

In [ ]:
# Create the estimator
xgb = sagemaker.estimator.Estimator(container,  # container uri
                                    role,
                                    # base_job_name='modelo-xgb',
                                    volume_size=8,
                                    instance_count=1,  # instances number
                                    instance_type='ml.m4.xlarge',  # instances type
                                    output_path='s3://{}/{}/output'.format(session.default_bucket(), s3_folder_bucket), # results storage
                                    sagemaker_session=session)

In [ ]:
print(f"URI de los resultados: {'s3://{}/{}/output'.format(session.default_bucket(), s3_folder_bucket)}")

In [ ]:
# Set the hyperparameters
xgb.set_hyperparameters(max_depth=5,
                        eta=0.2,
                        gamma=4,
                        min_child_weight=6,
                        subsample=0.8,
                        silent=0,
                        objective='binary:logistic',
                        early_stopping_rounds=12,
                        num_round=500)

### 2 - Entrenamiento

In [ ]:
inputs= {}

In [ ]:
# Load the data from S3, the data came from the previous notebook
s3_input_train = sagemaker.TrainingInput(s3_data=inputs['train'], content_type='csv')
s3_input_validation = sagemaker.TrainingInput(s3_data=inputs['validation'], content_type='csv')
s3_input_test = sagemaker.TrainingInput(s3_data=inputs['test'], content_type='csv')

In [ ]:
# Training
xgb.fit({'train': s3_input_train, 'validation': s3_input_validation})

### 3 - Deployment and endpoint creation

#### A - Deployment Serverless

In [ ]:
# We only pay for the usage we make of the model
from sagemaker.serverless import ServerlessInferenceConfig

In [ ]:
# endpoint configuration for serverless inference
serverless_config = ServerlessInferenceConfig(
  memory_size_in_mb=1024,
  max_concurrency=5,
)

In [ ]:
# deploy the model to an endpoint
xgb_predictor = xgb.deploy(endpoint_name='my-endpoint-serverless',
                           serverless_inference_config=serverless_config,
                           serializer=CSVSerializer())

#### B - Deploy in a VM

In [ ]:
# Code for deploying the model to a non-serverless endpoint, which is more expensive but has better performance.

# xgb_predictor = xgb.deploy(endpoint_name='my-endpoint',
#                            initial_instance_count=1,
#                            instance_type='ml.m4.xlarge',
#                            serializer=CSVSerializer())

### 4 - Hyperparameter optimization
<br> (Grid Search)

In [ ]:
from sagemaker.tuner import IntegerParameter, ContinuousParameter, HyperparameterTuner

xgb_hyperparameter_tuner = HyperparameterTuner(estimator = xgb, # Estimator
                                               objective_metric_name = 'validation:rmse', # Metric for compare models
                                               objective_type = 'Minimize', # Minimize or Maximize the metric
                                               max_jobs = 4, # Total number of models to train
                                               max_parallel_jobs = 2, # Number of models to train in parallel
                                               hyperparameter_ranges = {
                                                    'max_depth': IntegerParameter(3, 12),
                                                    'eta'      : ContinuousParameter(0.05, 0.5),
                                                    'min_child_weight': IntegerParameter(2, 8),
                                                    'subsample': ContinuousParameter(0.5, 0.9),
                                                    'gamma': ContinuousParameter(0, 10),
                                               })

In [ ]:
# Run the hyperparameter tuning job
# xgb_hyperparameter_tuner.fit({'train': s3_input_train, 'validation': s3_input_validation})